# 02 — RL Agent Training

Trains 4 agents per ticker (PPO and DQN, price-only vs price+sentiment):

| Agent | State |
|-------|-------|
| PPO   | price-only |
| PPO   | price + sentiment |
| DQN   | price-only |
| DQN   | price + sentiment |

Note: SAC needs a continuous action space so it doesn't work here — DQN handles our discrete (close/hold/buy/short) actions fine.

Train: Jul 1 – Oct 31, 2025 (approx 87 trading days, Q3 + first bit of Q4)
Test:  Nov 1 – Dec 31, 2025 (approx 41 trading days)

Models saved to `models/`.

In [1]:
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from trading_env import TradingEnv, make_envs

from stable_baselines3 import PPO, DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor

MODELS_DIR = PROJECT_ROOT / 'models'
DATA_DIR   = PROJECT_ROOT / 'data'
MODELS_DIR.mkdir(exist_ok=True)

print('Setup complete.')

Setup complete.


## 1. Load Dataset

In [2]:
daily       = pd.read_csv(DATA_DIR / 'daily_features.csv',         parse_dates=['date'])
daily_finbert = pd.read_csv(DATA_DIR / 'daily_features_finbert.csv', parse_dates=['date'])

TICKERS = ['NVDA', 'GOOG', 'TSLA']
TRAIN_CUTOFF = '2025-10-31'

print('VADER dataset shape:  ', daily.shape)
print('FinBERT dataset shape:', daily_finbert.shape)
print('Tickers:', daily['ticker'].unique())
print()

for t in TICKERS:
    sub = daily[daily['ticker'] == t]
    train = sub[sub['date'] <= TRAIN_CUTOFF]
    test  = sub[sub['date'] >  TRAIN_CUTOFF]
    print(f'{t}: {len(train)} train days, {len(test)} test days')

VADER dataset shape:   (384, 33)
FinBERT dataset shape: (384, 33)
Tickers: ['GOOG' 'NVDA' 'TSLA']

NVDA: 87 train days, 41 test days
GOOG: 87 train days, 41 test days
TSLA: 87 train days, 41 test days


## 2. Sanity Check — Environment

In [3]:
# leaving penalties at 0 for now — can tune these separately later
ENV_KWARGS = dict(
    reward_mode='log_return',
    drawdown_penalty=0.0,
    volatility_penalty=0.0,
    
    # added volatility threshold
    use_volatility_gate = True,
    vol_threshold_quantile = 0.3,
)

# quick check that the NVDA env looks right
envs = make_envs(daily, 'NVDA', TRAIN_CUTOFF, env_kwargs=ENV_KWARGS)

for env_key, label in [
    ('train_price', 'price-only env'),
    ('train_sentiment_basic', 'basic sentiment env'),
]:
    print(f'Checking {label}...')
    check_env(envs[env_key], warn=True)
    print('OK')

print(f"Obs dim (price-only):       {envs['train_price'].observation_space.shape[0]}")
print(f"Obs dim (sentiment_basic):  {envs['train_sentiment_basic'].observation_space.shape[0]}")


Checking price-only env...
OK
Checking basic sentiment env...
OK
Obs dim (price-only):       10
Obs dim (sentiment_basic):  17


## 3. Training Configuration

In [4]:
# 60k timesteps — gives us roughly 690 episodes over the ~87 day window
TOTAL_TIMESTEPS = 60_000

# keeping penalties at 0 for now
ENV_KWARGS = dict(
    reward_mode='log_return',
    drawdown_penalty=0.0,
    volatility_penalty=0.0,
    
    # added volatility threshold
    use_volatility_gate = True,
    vol_threshold_quantile = 0.3,
)

PPO_KWARGS = dict(
    policy='MlpPolicy',
    n_steps=87,         # one full episode per rollout (87 trading days)
    batch_size=29,      # must divide n_steps; 87 = 3 * 29
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    learning_rate=3e-4,
    verbose=0,
)

DQN_KWARGS = dict(
    policy='MlpPolicy',
    learning_rate=1e-3,
    buffer_size=10_000,
    learning_starts=87,   # one full episode before learning starts
    batch_size=32,
    tau=0.005,
    gamma=0.99,
    train_freq=4,
    gradient_steps=1,
    exploration_final_eps=0.05,
    exploration_fraction=0.3,
    verbose=0,
)

print(f'Training {TOTAL_TIMESTEPS:,} timesteps per agent.')
print('Environment kwargs:', ENV_KWARGS)
print('Configs set.')

Training 60,000 timesteps per agent.
Environment kwargs: {'reward_mode': 'log_return', 'drawdown_penalty': 0.0, 'volatility_penalty': 0.0, 'use_volatility_gate': True, 'vol_threshold_quantile': 0.3}
Configs set.


## 4. Train All Agents

4 agents × 3 tickers = 12 models total. At 60k timesteps each one takes roughly 90s (PPO) or ~45s (DQN).

In [5]:
from tqdm.notebook import tqdm

trained_models = {}  # key: (ticker, algo, state_type)

# price-only and vader both use the vader dataset
configs = [
    ('PPO', 'price',            PPO, PPO_KWARGS, 'train_price',            daily),
    ('PPO', 'sentiment_vader',  PPO, PPO_KWARGS, 'train_sentiment_basic',  daily),
    ('DQN', 'price',            DQN, DQN_KWARGS, 'train_price',            daily),
    ('DQN', 'sentiment_vader',  DQN, DQN_KWARGS, 'train_sentiment_basic',  daily),
    # finbert uses the same env structure but swaps in the finbert dataset
    ('PPO', 'sentiment_finbert', PPO, PPO_KWARGS, 'train_sentiment_basic', daily_finbert),
    ('DQN', 'sentiment_finbert', DQN, DQN_KWARGS, 'train_sentiment_basic', daily_finbert),
]

for ticker in TICKERS:
    print(f'\n=== {ticker} ===')

    for algo_name, state_type, AlgoClass, kwargs, env_key, dataset in tqdm(configs, desc=ticker):
        save_path = MODELS_DIR / f'{ticker}_{algo_name}_{state_type}'

        key = (ticker, algo_name, state_type)
        if save_path.with_suffix('.zip').exists():
            trained_models[key] = AlgoClass.load(str(save_path))
            print(f'  Loaded  {save_path.name} (already exists)')
            continue

        envs = make_envs(dataset, ticker, TRAIN_CUTOFF, env_kwargs=ENV_KWARGS)
        env  = Monitor(envs[env_key])
        model = AlgoClass(env=env, seed=42, **kwargs)
        model.learn(total_timesteps=TOTAL_TIMESTEPS)

        trained_models[key] = model
        model.save(str(save_path))
        print(f'  Saved   {save_path.name}')

print('\nAll models trained.')



=== NVDA ===


NVDA:   0%|          | 0/6 [00:00<?, ?it/s]

  Loaded  NVDA_PPO_price (already exists)
  Loaded  NVDA_PPO_sentiment_vader (already exists)
  Loaded  NVDA_DQN_price (already exists)
  Loaded  NVDA_DQN_sentiment_vader (already exists)
  Loaded  NVDA_PPO_sentiment_finbert (already exists)
  Loaded  NVDA_DQN_sentiment_finbert (already exists)

=== GOOG ===


GOOG:   0%|          | 0/6 [00:00<?, ?it/s]

  Loaded  GOOG_PPO_price (already exists)
  Loaded  GOOG_PPO_sentiment_vader (already exists)
  Loaded  GOOG_DQN_price (already exists)
  Saved   GOOG_DQN_sentiment_vader
  Saved   GOOG_PPO_sentiment_finbert
  Saved   GOOG_DQN_sentiment_finbert

=== TSLA ===


TSLA:   0%|          | 0/6 [00:00<?, ?it/s]

  Loaded  TSLA_PPO_price (already exists)
  Saved   TSLA_PPO_sentiment_vader
  Loaded  TSLA_DQN_price (already exists)
  Saved   TSLA_DQN_sentiment_vader
  Saved   TSLA_PPO_sentiment_finbert
  Saved   TSLA_DQN_sentiment_finbert

All models trained.


## 5. Quick Sanity Check

Just running each model on the test set to see final portfolio values — not the full eval, just making sure nothing is broken.

In [6]:
def evaluate_model(model, env):
    """Run one episode and return history + final portfolio value."""
    obs, _ = env.reset()
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, _, done, _, info = env.step(action)
    return env.get_history(), info['portfolio_value']

print(f'Initial cash: $10,000.00\n')
print(f'{"Ticker":<6} {"Algo":<5} {"State":<22} {"Final $":>10}')
print('-' * 50)

test_results = {}
for ticker in TICKERS:
    envs_vader   = make_envs(daily,         ticker, TRAIN_CUTOFF, env_kwargs=ENV_KWARGS)
    envs_finbert = make_envs(daily_finbert, ticker, TRAIN_CUTOFF, env_kwargs=ENV_KWARGS)

    for algo_name, state_type, _, _, env_key, _ in configs:
        test_env_key = env_key.replace('train_', 'test_')
        envs = envs_finbert if state_type == 'sentiment_finbert' else envs_vader
        key = (ticker, algo_name, state_type)
        model = trained_models[key]
        history, final_val = evaluate_model(model, envs[test_env_key])
        test_results[key] = (history, final_val)
        print(f'{ticker:<6} {algo_name:<5} {state_type:<22} ${final_val:>9.2f}')


Initial cash: $10,000.00

Ticker Algo  State                     Final $
--------------------------------------------------
NVDA   PPO   price                  $  9902.35
NVDA   PPO   sentiment_vader        $ 10321.72
NVDA   DQN   price                  $ 10716.97
NVDA   DQN   sentiment_vader        $ 10131.45
NVDA   PPO   sentiment_finbert      $ 10510.55
NVDA   DQN   sentiment_finbert      $ 10112.67
GOOG   PPO   price                  $ 10847.55
GOOG   PPO   sentiment_vader        $ 10598.78
GOOG   DQN   price                  $ 10769.31
GOOG   DQN   sentiment_vader        $ 10130.78
GOOG   PPO   sentiment_finbert      $ 10494.05
GOOG   DQN   sentiment_finbert      $ 10344.97
TSLA   PPO   price                  $ 10446.97
TSLA   PPO   sentiment_vader        $ 10193.42
TSLA   DQN   price                  $ 10615.63
TSLA   DQN   sentiment_vader        $ 10207.67
TSLA   PPO   sentiment_finbert      $ 10065.93
TSLA   DQN   sentiment_finbert      $  9753.17


## 6. Short-Selling Extension

Adds 4 more agents per ticker with an expanded 4-action space:

| Action | Meaning |
|--------|---------|
| 0 | Close all — sell longs, cover shorts |
| 1 | Hold |
| 2 | Buy long (covers shorts first) |
| 3 | Short (closes longs first) |

Lets the agent profit on bad-sentiment days instead of just going to cash.
12 new models total (PPO/DQN × price/sentiment × 3 tickers), saved as `{TICKER}_{ALGO}_{state}_short.zip`.

In [7]:
configs_short = [
    ('PPO', 'price_short',            PPO, PPO_KWARGS, 'train_price_short',           daily),
    ('PPO', 'sentiment_vader_short',  PPO, PPO_KWARGS, 'train_sentiment_basic_short', daily),
    ('DQN', 'price_short',            DQN, DQN_KWARGS, 'train_price_short',           daily),
    ('DQN', 'sentiment_vader_short',  DQN, DQN_KWARGS, 'train_sentiment_basic_short', daily),
    ('PPO', 'sentiment_finbert_short', PPO, PPO_KWARGS, 'train_sentiment_basic_short', daily_finbert),
    ('DQN', 'sentiment_finbert_short', DQN, DQN_KWARGS, 'train_sentiment_basic_short', daily_finbert),
]

for ticker in TICKERS:
    print(f'\n=== {ticker} (short-enabled) ===')

    for algo_name, state_type, AlgoClass, kwargs, env_key, dataset in tqdm(configs_short, desc=ticker):
        save_path = MODELS_DIR / f'{ticker}_{algo_name}_{state_type}'

        key = (ticker, algo_name, state_type)
        if save_path.with_suffix('.zip').exists():
            trained_models[key] = AlgoClass.load(str(save_path))
            print(f'  Loaded  {save_path.name} (already exists)')
            continue

        envs = make_envs(dataset, ticker, TRAIN_CUTOFF, env_kwargs=ENV_KWARGS, allow_short=True)
        env  = Monitor(envs[env_key])
        model = AlgoClass(env=env, seed=42, **kwargs)
        model.learn(total_timesteps=TOTAL_TIMESTEPS)

        trained_models[key] = model
        model.save(str(save_path))
        print(f'  Saved   {save_path.name}')

print('\nAll short-enabled models trained.')



=== NVDA (short-enabled) ===


NVDA:   0%|          | 0/6 [00:00<?, ?it/s]

  Loaded  NVDA_PPO_price_short (already exists)
  Saved   NVDA_PPO_sentiment_vader_short
  Loaded  NVDA_DQN_price_short (already exists)
  Saved   NVDA_DQN_sentiment_vader_short
  Saved   NVDA_PPO_sentiment_finbert_short
  Saved   NVDA_DQN_sentiment_finbert_short

=== GOOG (short-enabled) ===


GOOG:   0%|          | 0/6 [00:00<?, ?it/s]

  Loaded  GOOG_PPO_price_short (already exists)
  Saved   GOOG_PPO_sentiment_vader_short
  Loaded  GOOG_DQN_price_short (already exists)
  Saved   GOOG_DQN_sentiment_vader_short
  Saved   GOOG_PPO_sentiment_finbert_short
  Saved   GOOG_DQN_sentiment_finbert_short

=== TSLA (short-enabled) ===


TSLA:   0%|          | 0/6 [00:00<?, ?it/s]

  Loaded  TSLA_PPO_price_short (already exists)
  Saved   TSLA_PPO_sentiment_vader_short
  Loaded  TSLA_DQN_price_short (already exists)
  Saved   TSLA_DQN_sentiment_vader_short
  Saved   TSLA_PPO_sentiment_finbert_short
  Saved   TSLA_DQN_sentiment_finbert_short

All short-enabled models trained.


In [8]:
print(f'Initial cash: $10,000.00\n')
print(f'{"Ticker":<6} {"Algo":<5} {"State":<28} {"Final $":>10}  {"Short days":>11}')
print('-' * 65)

for ticker in TICKERS:
    envs_vader   = make_envs(daily,         ticker, TRAIN_CUTOFF, env_kwargs=ENV_KWARGS, allow_short=True)
    envs_finbert = make_envs(daily_finbert, ticker, TRAIN_CUTOFF, env_kwargs=ENV_KWARGS, allow_short=True)

    for algo_name, state_type, _, _, env_key, _ in configs_short:
        test_env_key = env_key.replace('train_', 'test_')
        envs = envs_finbert if 'finbert' in state_type else envs_vader
        key = (ticker, algo_name, state_type)
        model = trained_models[key]
        history, final_val = evaluate_model(model, envs[test_env_key])
        short_days = int((history['action'] == 3).sum())
        print(f'{ticker:<6} {algo_name:<5} {state_type:<28} ${final_val:>9.2f}  {short_days:>11}')

Initial cash: $10,000.00

Ticker Algo  State                           Final $   Short days
-----------------------------------------------------------------
NVDA   PPO   price_short                  $ 11296.74           11
NVDA   PPO   sentiment_vader_short        $ 11058.16            6
NVDA   DQN   price_short                  $ 11016.44           16
NVDA   DQN   sentiment_vader_short        $ 10753.70           25
NVDA   PPO   sentiment_finbert_short      $ 10738.00            6
NVDA   DQN   sentiment_finbert_short      $ 11761.26           23
GOOG   PPO   price_short                  $ 11002.09           17
GOOG   PPO   sentiment_vader_short        $ 10569.09            0
GOOG   DQN   price_short                  $  9311.19           22
GOOG   DQN   sentiment_vader_short        $ 10098.18           23
GOOG   PPO   sentiment_finbert_short      $ 11143.40            5
GOOG   DQN   sentiment_finbert_short      $ 10096.98           29
TSLA   PPO   price_short                  $ 11560.